In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("../data/ai4i2020.csv")

df.head()
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
df.isnull().sum()

Shape: (10000, 14)

Columns:
['UDI', 'Product ID', 'Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']


UDI                        0
Product ID                 0
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
RNF                        0
dtype: int64

In [3]:
df.isnull()
df["Machine failure"].value_counts()
df["Machine failure"].value_counts(normalize=True) * 100

Machine failure
0    96.61
1     3.39
Name: proportion, dtype: float64

In [4]:
df.columns.tolist()

['UDI',
 'Product ID',
 'Type',
 'Air temperature [K]',
 'Process temperature [K]',
 'Rotational speed [rpm]',
 'Torque [Nm]',
 'Tool wear [min]',
 'Machine failure',
 'TWF',
 'HDF',
 'PWF',
 'OSF',
 'RNF']

In [5]:

print("Machine Types:")
print(df["Type"].value_counts())


print("\nNumerical Feature Summary:")
print(df.describe())


print("\nUnique Types:")
print(df["Type"].unique())


print("\nMachine Failure Counts:")
print(df["Machine failure"].value_counts())

print("\nMachine Failure Percentage:")
print(df["Machine failure"].value_counts(normalize=True) * 100)

Machine Types:
Type
L    6000
M    2997
H    1003
Name: count, dtype: int64

Numerical Feature Summary:
               UDI  Air temperature [K]  Process temperature [K]  \
count  10000.00000         10000.000000             10000.000000   
mean    5000.50000           300.004930               310.005560   
std     2886.89568             2.000259                 1.483734   
min        1.00000           295.300000               305.700000   
25%     2500.75000           298.300000               308.800000   
50%     5000.50000           300.100000               310.100000   
75%     7500.25000           301.500000               311.100000   
max    10000.00000           304.500000               313.800000   

       Rotational speed [rpm]   Torque [Nm]  Tool wear [min]  Machine failure  \
count            10000.000000  10000.000000     10000.000000     10000.000000   
mean              1538.776100     39.986910       107.951000         0.033900   
std                179.284096      9.968

In [6]:
# Select the features that will be used to predict machine failure
feature_columns = [
    "Type",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

# Create feature matrix X
X = df[feature_columns]

# Create target variable y
y = df["Machine failure"]

# Inspect the selected features
print("X Shape:", X.shape)
print("y Shape:", y.shape)

print("\nSelected Features:")
print(X.head())

print("\nTarget:")
print(y.head())

X Shape: (10000, 6)
y Shape: (10000,)

Selected Features:
  Type  Air temperature [K]  Process temperature [K]  Rotational speed [rpm]  \
0    M                298.1                    308.6                    1551   
1    L                298.2                    308.7                    1408   
2    L                298.1                    308.5                    1498   
3    L                298.2                    308.6                    1433   
4    L                298.2                    308.7                    1408   

   Torque [Nm]  Tool wear [min]  
0         42.8                0  
1         46.3                3  
2         49.4                5  
3         39.5                7  
4         40.0                9  

Target:
0    0
1    0
2    0
3    0
4    0
Name: Machine failure, dtype: int64


In [7]:

X_encoded = pd.get_dummies(X, columns=["Type"], dtype=int)


print("Original X shape:", X.shape)
print("Encoded X shape:", X_encoded.shape)

print("\nEncoded Columns:")
print(X_encoded.columns.tolist())

print("\nFirst 5 Rows:")
print(X_encoded.head())

Original X shape: (10000, 6)
Encoded X shape: (10000, 8)

Encoded Columns:
['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Type_H', 'Type_L', 'Type_M']

First 5 Rows:
   Air temperature [K]  Process temperature [K]  Rotational speed [rpm]  \
0                298.1                    308.6                    1551   
1                298.2                    308.7                    1408   
2                298.1                    308.5                    1498   
3                298.2                    308.6                    1433   
4                298.2                    308.7                    1408   

   Torque [Nm]  Tool wear [min]  Type_H  Type_L  Type_M  
0         42.8                0       0       0       1  
1         46.3                3       0       1       0  
2         49.4                5       0       1       0  
3         39.5                7       0       1       0  
4         40.0              

In [8]:
from sklearn.model_selection import train_test_split

# Split 
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True) * 100)

X_train shape: (8000, 8)
X_test shape: (2000, 8)
y_train shape: (8000,)
y_test shape: (2000,)

Training target distribution:
Machine failure
0    96.6125
1     3.3875
Name: proportion, dtype: float64

Testing target distribution:
Machine failure
0    96.6
1     3.4
Name: proportion, dtype: float64


In [9]:
from sklearn.linear_model import LogisticRegression

# Create the baseline Logistic Regression model
logistic_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

# Train the model using only the training data
logistic_model.fit(X_train, y_train)

print("Logistic Regression training complete.")

Logistic Regression training complete.


In [10]:
# Predict classes for unseen test data
y_pred = logistic_model.predict(X_test)

# Predict failure probabilities
y_prob = logistic_model.predict_proba(X_test)[:, 1]

print("First 10 actual values:")
print(y_test.iloc[:10].values)

print("\nFirst 10 predicted values:")
print(y_pred[:10])

print("\nFirst 10 failure probabilities:")
print(y_prob[:10])

First 10 actual values:
[0 0 0 0 0 0 0 0 0 0]

First 10 predicted values:
[1 0 1 0 1 1 1 0 0 1]

First 10 failure probabilities:
[0.97334781 0.48898306 0.66148915 0.01498411 0.85509914 0.65047181
 0.80535524 0.13892769 0.02805172 0.58606767]


In [11]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Logistic Regression Evaluation")
print("--------------------------------")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Logistic Regression Evaluation
--------------------------------
Accuracy:  0.8270
Precision: 0.1436
Recall:    0.8235
F1 Score:  0.2445

Confusion Matrix:
[[1598  334]
 [  12   56]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.83      0.90      1932
           1       0.14      0.82      0.24        68

    accuracy                           0.83      2000
   macro avg       0.57      0.83      0.57      2000
weighted avg       0.96      0.83      0.88      2000

